# Exercises - The Carry Trade

#### Notation Commands

$$\newcommand{\Black}{\mathcal{B}}
\newcommand{\Blackcall}{\Black_{\mathrm{call}}}
\newcommand{\Blackput}{\Black_{\mathrm{put}}}
\newcommand{\EcondS}{\hat{S}_{\mathrm{conditional}}}
\newcommand{\Efwd}{\mathbb{E}^{T}}
\newcommand{\Ern}{\mathbb{E}^{\mathbb{Q}}}
\newcommand{\Tfwd}{T_{\mathrm{fwd}}}
\newcommand{\Tunder}{T_{\mathrm{bond}}}
\newcommand{\accint}{A}
\newcommand{\carry}{\widetilde{\cpn}}
\newcommand{\cashflow}{C}
\newcommand{\convert}{\phi}
\newcommand{\cpn}{c}
\newcommand{\ctd}{\mathrm{CTD}}
\newcommand{\disc}{Z}
\newcommand{\done}{d_{1}}
\newcommand{\dt}{\Delta t}
\newcommand{\dtwo}{d_{2}}
\newcommand{\flatvol}{\sigma_{\mathrm{flat}}}
\newcommand{\flatvolT}{\sigma_{\mathrm{flat},T}}
\newcommand{\float}{\mathrm{flt}}
\newcommand{\freq}{m}
\newcommand{\futprice}{\mathcal{F}(t,T)}
\newcommand{\futpriceDT}{\mathcal{F}(t+h,T)}
\newcommand{\futpriceT}{\mathcal{F}(T,T)}
\newcommand{\futrate}{\mathscr{f}}
\newcommand{\fwdprice}{F(t,T)}
\newcommand{\fwdpriceDT}{F(t+h,T)}
\newcommand{\fwdpriceT}{F(T,T)}
\newcommand{\fwdrate}{f}
\newcommand{\fwdvol}{\sigma_{\mathrm{fwd}}}
\newcommand{\fwdvolTi}{\sigma_{\mathrm{fwd},T_i}}
\newcommand{\grossbasis}{B}
\newcommand{\hedge}{\Delta}
\newcommand{\ivol}{\sigma_{\mathrm{imp}}}
\newcommand{\logprice}{p}
\newcommand{\logyield}{y}
\newcommand{\mat}{(n)}
\newcommand{\nargcond}{d_{1}}
\newcommand{\nargexer}{d_{2}}
\newcommand{\netbasis}{\tilde{\grossbasis}}
\newcommand{\normcdf}{\mathcal{N}}
\newcommand{\notional}{K}
\newcommand{\pfwd}{P_{\mathrm{fwd}}}
\newcommand{\pnl}{\Pi}
\newcommand{\price}{P}
\newcommand{\probexer}{\hat{\mathcal{P}}_{\mathrm{exercise}}}
\newcommand{\pvstrike}{K^*}
\newcommand{\refrate}{r^{\mathrm{ref}}}
\newcommand{\rrepo}{r^{\mathrm{repo}}}
\newcommand{\spotrate}{r}
\newcommand{\spread}{s}
\newcommand{\strike}{K}
\newcommand{\swap}{\mathrm{sw}}
\newcommand{\swaprate}{\cpn_{\swap}}
\newcommand{\tbond}{\mathrm{fix}}
\newcommand{\ttm}{\tau}
\newcommand{\value}{V}
\newcommand{\vega}{\nu}
\newcommand{\years}{\tau}
\newcommand{\yearsACT}{\tau_{\mathrm{act/360}}}
\newcommand{\yield}{Y}$$

Use the data set `famabliss_strips_2025-11-28.xlsx`.

It gives prices on **zero coupon bonds** with maturities of 1 through 5 years.
* These are prices per \$1 face value on bonds that only pay principal.
* Such bonds can be created from treasuries by *stripping* out their coupons.
* In essence, you can consider these prices as the discount factors $Z$, for maturity intervals 1 through 5 years.

In this problem, we focus on six dates: the month of **November** in 2020 through 2025.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

SHEET_NAME = 'famabliss_strips_2025-11-28.xlsx'
df = pd.read_excel(SHEET_NAME)
df['date'] = pd.to_datetime(df['date'])

nov_df = df[(df['date'].dt.month == 11) & (df['date'].dt.year >= 2020) & (df['date'].dt.year <= 2025)].copy()
display(nov_df)

,date,1,2,3,4,5
821,2020-11-30,0.998839,0.997047,0.994464,0.988809,0.981515
833,2021-11-30,0.997523,0.988614,0.974796,0.958322,0.943772
845,2022-11-30,0.954375,0.918220,0.887608,0.857356,0.830593
857,2023-11-30,0.951080,0.911951,0.876957,0.842191,0.809378
869,2024-11-29,0.959187,0.920923,0.885440,0.850418,0.817795
881,2025-11-28,0.964936,0.932866,0.900919,0.868240,0.836057


# The Carry Trade

## 1.1

Suppose it is `November 2020`, and you determine to implement a carry trade with the following specification...

* Long `$100` million (market value, not face value) of the 5-year zero-coupon bond (maturing `Nov 2025`.)
* Short `$100` million (market value, not face value) of the 1-year zero-coupon bond (maturing `Nov 2021`.)
* Assume there is a `2%` haircut on each side of the trade, so it requires `$4` million of investor capital to initiate it.

1. Calculate the total profit and loss year-by-year.
1. Calculate the total return (`Nov 2025`) on the initial \\$4 million of investor capital.

#### Short position
* Each year you will roll over the short position to maintain a short `$100` million (market value) in the 1-year bond.
* This will require injecting more cash into the trade, as the expiring short will require more than `$100` million to close out. 
* In `Nov 2024`, no need to open a new short position, as your long position will (at that point) be a one-year bond.

#### Alternatives
The scheme above is for simplicity. You could try more interesting ways of setting the short position...
* Open a new short position sized to whatever is needed to cover the expiring short position.
* Set the short positions to duration-hedge the long position.

In [2]:
nov = nov_df.set_index("date").sort_index()
years = nov.index.year.tolist()

MV = 100_000_000
CAPITAL = 4_000_000
HAIRCUT = 0.02
INITIAL_CAPITAL = HAIRCUT * MV * 2

# Face value of long bond
face_long = MV / nov.iloc[0][5]

rows = []

prev_long_value = MV
face_short = 0
capital = INITIAL_CAPITAL

for i, date in enumerate(nov.index):
    year = date.year

    # Long Position
    if i == 0:
        long_value = MV
    elif i < 5:
        remaining = 5 - i
        long_value = face_long * nov.loc[date, remaining]
    else:
        long_value = face_long * 1.0

    long_pnl = long_value - prev_long_value

    # Short Position
    if i == 0:
        short_pnl = 0.0    
    else:
        close_cf = -face_short
        open_cf = MV if year <= 2023 else 0.0
        short_cf = open_cf + close_cf
        injection = max(0, -short_cf)
        capital += injection
    if year <= 2023:
        z1 = nov.loc[date, 1]
        face_short = MV / z1
    else:
        face_short = 0.0

    total_pnl = long_pnl + short_pnl

    rows.append({
        "Year": year,
        "Long Value ($)": long_value,
        "Long P&L ($)": long_pnl,
        "Short Cash Flow ($)": short_pnl,
        "Total P&L ($)": total_pnl,
        "Capital ($)": capital,
    })

    prev_long_value = long_value

pnl_df = pd.DataFrame(rows)
display(pnl_df)


,Year,Long Value ($),Long P&L ($),Short Cash Flow ($),Total P&L ($),Capital ($)
0,2020,1.000000e+08,0.000000e+00,0.0,0.000000e+00,4.000000e+06
1,2021,9.763703e+07,-2.362973e+06,0.0,-2.362973e+06,4.116228e+06
2,2022,9.043237e+07,-7.204655e+06,0.0,-7.204655e+06,4.364575e+06
3,2023,9.291255e+07,2.480175e+06,0.0,2.480175e+06,9.145191e+06
4,2024,9.772511e+07,4.812566e+06,0.0,4.812566e+06,1.142888e+08
5,2025,1.018833e+08,4.158168e+06,0.0,4.158168e+06,1.142888e+08


In [3]:
m = pnl_df.copy()

# Capital at start of year (previous row's capital)
m["Capital Start ($)"] = m["Capital ($)"].shift(1)
m.loc[m["Year"] == 2020, "Capital Start ($)"] = np.nan  # initiation

# Yearly returns on capital actually tied up
m["rets"] = m["Total P&L ($)"] / m["Capital Start ($)"]

# Cumulative return on INITIAL capital (as in your professor screenshot label)
m["cumulative ret (on initial capital)"] = m["Total P&L ($)"].cumsum() / INITIAL_CAPITAL

# Initiation row is NaN for returns
m.loc[m["Year"] == 2020, ["rets", "cumulative ret (on initial capital)"]] = np.nan

out = m[["Year", "Capital ($)", "rets", "cumulative ret (on initial capital)"]]

display(out.style.format({
    "Capital ($)": "{:.6e}",
    "rets": "{:.6f}",
    "cumulative ret (on initial capital)": "{:.6f}",
}))


,Year,Capital ($),rets,cumulative ret (on initial capital)
0,2020,4.000000e+06,nan,nan
1,2021,4.116228e+06,-0.590743,-0.590743
2,2022,4.364575e+06,-1.750305,-2.391907
3,2023,9.145191e+06,0.568251,-1.771863
4,2024,1.142888e+08,0.526240,-0.568722
5,2025,1.142888e+08,0.036383,0.470820


## 1.2

How would this trade play out if the path of one-year spot rates equaled the forward rates observed in `2020`?

If the realized path of one-year spot rates exactly equaled the forward rates observed in November 2020, then repeating the carry-trade calculation in 1.1 with those forward-implied rates as inputs would produce a different realized P&L path than what we observed historically.

The rollover costs on the short position would unfold exactly as implied by the initial yield curve, eliminating unexpected increases in funding costs. As a result, the trade’s profitability would largely reflect what was already priced into the curve at inception, rather than gains or losses from unanticipated movements in short rates.

## 1.3

Given Fact 3 of the *dynamic* (conditional) tests of the Expectations Hypothesis (EH), do you expect that as of `Nov 2025` the long-short trade above looks more or less favorable for `Nov 2025-2030` than it did for `Nov 2020-2025`?

Fact 3 of the dynamic tests of the Expectations Hypothesis implies that expected excess returns on bonds vary over time and are predictable using forward spreads. Since the long–short carry trade’s expected excess return is directly related to the forward spread between longer-maturity forwards and the one-year yield, the attractiveness of the trade for Nov 2025–2030 depends on the level of forward spreads observed in Nov 2025. If forward spreads in 2025 are wider (narrower) than in 2020, the trade should look more (less) favorable going forward, reflecting time-variation in expected excess returns.

***